In [1]:
import os
import sys
import shutil
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import pandas as pd
import numpy as np
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from torch.utils.data import DataLoader

In [31]:
script_name = source_path+"/scripts/torch_train_on_rep.py"
kind = 'patches_224'#'body'
selected_FEs = ['clip-vit-large-patch14-inter','clip-vit-large-patch14-un','BEiT-Large','BEiT-Large-inter','trocr-large-handwritten-inter',
                'trocr-large-handwritten','convnext_large','convnext_large-inter','DeiT-Tiny-inter','DeiT-Tiny'
                ,'dresnet50-inter','dresnet50','crnn_vgg16_bn_224-inter','swin_s' , 'DeiT-Small','crnn_vgg16_bn_224'] # you can even take the names directly from the folders
#aggiungi swin_s , DeiT-Small, 'linknet_resnet18' 
selected_FEs = ['clip-vit-large-patch14-inter']
#0.0001	0.039003	0.764085 clip-inter-patches224
#with add aggregation: 0.0001	0.060253	0.757042
#with concat aggregation: 0.0001	0.082199	0.778169
#concat+mean: 0.0001	0.024798	0.746479 (notice acc weighted=acc single because i amm aggregating all in a page)
#add+mean: 0.0001	0.0113	0.71831
configs = [
    {'penalty':'l2', 'C':0.00001, 'solver':'lbfgs', 'max_iter':500,'hidden_layer_sizes':16},
    #{'penalty':'l2', 'C':0.001, 'solver':'lbfgs', 'max_iter':500,'hidden_layer_sizes':16},
    #{'penalty':'l2', 'C':0.01, 'solver':'lbfgs', 'max_iter':500,'hidden_layer_sizes':16},
    #{'penalty':'l2', 'C':0.1, 'solver':'lbfgs', 'max_iter':500},
    #{'penalty':'l2', 'C':1.0, 'solver':'lbfgs', 'max_iter':500}
]

In [32]:
#csv_location = 'icdar_EXTRACTED_train_df_clip-vit-large-patch14_20250517_144404.csv'
#parameters
import json
accuracies=[]
cross_vals=[]
script_name = source_path+"/scripts/sklearn_classifier_on_rep.py"
for selected_FE in selected_FEs:
    print(f"Evaluating feature extractor: {selected_FE}")
    for config in configs:
        print(f"Using configuration: {config}")
        data_augmentation = False
        suffix = '_augmented' if data_augmentation else ''
        train_filename,val_filename,_ = file_IO.load_input_files(source_path,selected_FE,kind,suffix)
        extra_view=True # works only with validation_mode=='val_only'
        extra_integration_mode = 'concat'  # 'concat' or 'add'
        aggregation_mode = None  # 'mean' or 'max' or None
        if extra_view:
            extra_train_filename, extra_val_filename, _ = file_IO.load_input_files(source_path,selected_FE,kind='body',suffix='')
        else:
            extra_train_filename, extra_val_filename = None, None

        selected_classifier='logreg'#'logreg'
        sklearn_model_parameters = config
        #solver='saga' for l1, 'lbfgs' for l2
        #c=0.01,0.1,1.0
        validation_mode = 'val_only' #'kfold_train_only' #'val_only', #1fold_train_only
        save_path = source_path+f'\\outputs\\online_deep_feature_extraction\\{selected_FE}\\representation_extraction\\sklearn_model_trained_on_rep\\{selected_classifier}'
        file_IO.access_or_create_dir(save_path)
        args = script_launching.DotDict(
            n_patches = -1,
            n_writers = -1,
            train_filename = train_filename,
            val_filename = val_filename,
            selected_model = selected_classifier,
            is_kaggle = False,
            with_pca = False,
            n_components = 0.95,
            validation_mode = validation_mode,  # 'kfold_train_only', 'val_only', '1fold_train_only'
            task = 'gender_detection',
            train_on_language = 'all',
            train_on_same = 'all',
            n_splits = 5,
            save_path = save_path,
            patch_merging = -1,
            data_augmentation = data_augmentation,
            extra_view=extra_view,
            extra_integration_mode = extra_integration_mode,
            extra_train_filename = extra_train_filename,
            extra_val_filename = extra_val_filename,
            aggregation_mode = aggregation_mode,
            sklearn_model_parameters = sklearn_model_parameters,
        )
        file_IO.save_args(args,save_path)  # Save the arguments to a file
        script_launching.run_experiment_threaded(args,script_name)  # Test a single run first
        
        with open(os.path.join(save_path, "best_model_performance_temp.json"), "r") as f:
            best_model_performance = json.load(f)

        acc_w=best_model_performance['average_ensembled_weighted_accuracy'] 
        accuracies.append(acc_w)
        cross_vals.append(best_model_performance['cross_val_accuracies'])
        print("="*50)
#best l2 0.6667, OOF Accuracy: 0.7183 ; c=0.01


#clipvit mlp-256 pca -> OOF Accuracy: 0.6754, OOF Accuracy: 0.7746
#nopca mlp-256 OOF Accuracy: 0.6839, OOF Accuracy: 0.7817
#mlp-64 pca OOF Accuracy: 0.6821, OOF Accuracy: 0.7746
#mlp-64 nopca OOF Accuracy: 0.6870, OOF Accuracy: 0.7887

Evaluating feature extractor: clip-vit-large-patch14-inter
Using configuration: {'penalty': 'l2', 'C': 1e-05, 'solver': 'lbfgs', 'max_iter': 500, 'hidden_layer_sizes': 16}
Starting experiment:
[STDOUT] Running feature extraction script...
[STDOUT] merging dfs train_1 and train_2 with lengths: 45120 1128
[STDOUT] Merged DataFrame length: 45120
[STDOUT] merging dfs train_1 and train_2 with lengths: 11360 284
[STDOUT] Merged DataFrame length: 11360
[STDOUT] aggregating patches, length before: 45120
[STDOUT] length after: 45120
[STDOUT] aggregating patches, length before: 11360
[STDOUT] length after: 11360
[STDOUT] Starting model cross-val...
[STDOUT] Fold 1 - IF Accuracy: 0.7877, IF Accuracy: 0.7970
[STDOUT] Fold 1 - OOF Accuracy: 0.7533, OOF Accuracy: 0.7606
[STDOUT] Average ensembled weighted accuracy: 0.7606
[STDOUT] Average individual accuracy: 0.7533
[STDOUT] Time taken to cross-validate the model: 8.66 seconds
[STDOUT] Model pipeline saved to file
[STDOUT] Best model performance sav

In [28]:
#print(cross_vals)
IF_acc=[c['IF'][0]['individual'] for c in cross_vals]
OOF_acc=[c['OOF'][0]['individual'] for c in cross_vals]
gen=[IF-OOF for IF,OOF in zip(IF_acc,OOF_acc)]

In [29]:
# Create a dataframe from model names and accuracies
unique_experiment = []
model_names = []
model_configs = []
for i in range(len(selected_FEs)):
    for j in range(len(configs)):
        unique_experiment.append(f"Exp_{i}_{j}")
        model_names.append(selected_FEs[i])
        model_configs.append(configs[j]['C'])
model_results_df = pd.DataFrame({
    'Experiment': unique_experiment,
    'Model': model_names,
    'Configuration': model_configs,
    'gen': gen,
    'Accuracy': accuracies
})

In [30]:
display(model_results_df[model_results_df['gen']<=0.1].sort_values(by='Accuracy'))

,Experiment,Model,Configuration,gen,Accuracy
0,Exp_0_0,clip-vit-large-patch14-inter,0.0001,0.0113,0.71831


# reload

In [13]:
def reload_modules():
    import importlib
    import utils.data_loading as data_loading
    import utils.visualization as visualization
    import utils.dataframes as dataframes
    import utils.utils_transforms as u_transforms
    import utils.training_utils as training_utils
    import utils.model_utils as model_utils
    import utils.file_IO as file_IO
    import utils.vit_rollout_mod as vit_rollout_mod
    import utils.script_launching as script_launching
    
    importlib.reload(file_IO)
    importlib.reload(data_loading)
    importlib.reload(visualization)
    importlib.reload(dataframes)
    importlib.reload(u_transforms)
    importlib.reload(model_utils)
    importlib.reload(training_utils)
    importlib.reload(vit_rollout_mod)
    importlib.reload(script_launching)

    return data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, script_launching
data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, script_launching = reload_modules()